## データ前処理
- bronze
- silver

### ブロンズ

In [0]:
CATALOG = "workspace"
SCHEMA = "bank"

CUSTOMER_PATH = "/Volumes/workspace/bank/vol/rdb/customer.csv"
TRANSACTION_PATH = "/Volumes/workspace/bank/vol/rdb/transaction_summary.csv"
CRM_PATH = "/Volumes/workspace/bank/vol/crm/crm_activity.csv"

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/bank/vol/rdb"))
display(dbutils.fs.ls("/Volumes/workspace/bank/vol/crm"))

In [0]:
bronze_customer_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CUSTOMER_PATH)
)

display(bronze_customer_df)

In [0]:
(
    bronze_customer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bank.bronze_customer")
)

In [0]:
bronze_transaction_summary_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(TRANSACTION_PATH)
)

display(bronze_transaction_summary_df)

In [0]:
(
    bronze_transaction_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bank.bronze_transaction_summary")
)

In [0]:
bronze_crm_activity_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CRM_PATH)
)

display(bronze_crm_activity_df)

In [0]:
(
    bronze_crm_activity_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bank.bronze_crm_activity")
)

In [0]:
%sql
SHOW TABLES IN workspace.bank;

In [0]:
%sql
SELECT
    'bronze_customer' AS table_name,
    COUNT(*) AS row_count
FROM workspace.bank.bronze_customer

UNION ALL

SELECT
    'bronze_transaction_summary',
    COUNT(*)
FROM workspace.bank.bronze_transaction_summary

UNION ALL

SELECT
    'bronze_crm_activity',
    COUNT(*)
FROM workspace.bank.bronze_crm_activity;

### シルバー

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, IntegerType

bronze_customer_df = spark.table(
    "workspace.bank.bronze_customer"
)

In [0]:
silver_customer_df = (
    bronze_customer_df
    .select(
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.trim(F.col("company_name")).alias("company_name"),
        F.trim(F.col("industry")).alias("industry"),
        F.col("annual_sales").cast(LongType()).alias("annual_sales"),
        F.col("employee_count").cast(IntegerType()).alias("employee_count"),
        F.trim(F.col("branch_name")).alias("branch_name"),
        F.trim(F.col("relationship_manager")).alias(
            "relationship_manager"
        ),
    )
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

In [0]:
(
    silver_customer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bank.silver_customer")
)

In [0]:
bronze_transaction_df = spark.table(
    "workspace.bank.bronze_transaction_summary"
)

In [0]:
silver_transaction_monthly_df = (
    bronze_transaction_df
    .select(
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.col("month").alias("transaction_month"),
        F.col("monthly_inflow").cast("long").alias("monthly_inflow"),
        F.col("monthly_outflow").cast("long").alias("monthly_outflow"),
        F.col("deposit_balance").cast("long").alias("deposit_balance"),
        F.col("loan_balance").cast("long").alias("loan_balance"),
        F.col("overdue_days").cast("int").alias("overdue_days"),
    )
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("transaction_month").isNotNull())
    .dropDuplicates([
        "customer_id",
        "transaction_month"
    ])
)

In [0]:
(
    silver_transaction_monthly_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.bank.silver_transaction_monthly"
    )
)

In [0]:
bronze_crm_df = spark.table(
    "workspace.bank.bronze_crm_activity"
)

In [0]:
silver_crm_activity_df = (
    bronze_crm_df
    .select(
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.to_date(F.col("activity_date")).alias("activity_date"),
        F.trim(F.col("meeting_note")).alias("meeting_note"),
        F.trim(F.col("customer_concern")).alias(
            "customer_concern"
        ),
        F.trim(F.col("next_action")).alias("next_action"),
    )
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("activity_date").isNotNull())
    .dropDuplicates(
        ["customer_id", "activity_date", "meeting_note"]
    )
)

In [0]:
(
    silver_crm_activity_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bank.silver_crm_activity")
)

In [0]:
%sql
SELECT
    'silver_customer' AS table_name,
    COUNT(*) AS row_count
FROM workspace.bank.silver_customer

UNION ALL

SELECT
    'silver_transaction_monthly',
    COUNT(*)
FROM workspace.bank.silver_transaction_monthly

UNION ALL

SELECT
    'silver_crm_activity',
    COUNT(*)
FROM workspace.bank.silver_crm_activity;